In [3]:
# ! uv pip install sqlalchemy psycopg2-binary pyarrow

In [8]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.exc import OperationalError

In [5]:
# ---- Paths ----
ROOT_DATA_DIR = Path.cwd().parent.parent / "dataset"
INDIVIDUAL_DATA_DIR = ROOT_DATA_DIR / "which_vlm_data" / "individual_datasets"

In [6]:
# Collect all parquet files (adjust pattern if needed)
PARQUET_FILES = sorted(INDIVIDUAL_DATA_DIR.glob("*.parquet"))
print(f"Found {len(PARQUET_FILES)} parquet files:")
for f in PARQUET_FILES:
    print(" -", f.name)


Found 50 parquet files:
 - ai2d.parquet
 - all_results.parquet
 - aokvqa.parquet
 - chart2text.parquet
 - chartqa.parquet
 - clevr.parquet
 - cocoqa.parquet
 - datikz.parquet
 - diagram_image_to_text.parquet
 - docvqa.parquet
 - dvqa.parquet
 - figureqa.parquet
 - finqa.parquet
 - geomverse.parquet
 - hateful_memes.parquet
 - hitab.parquet
 - iam.parquet
 - iconqa.parquet
 - infographic_vqa.parquet
 - intergps.parquet
 - localized_narratives.parquet
 - mapqa.parquet
 - mimic_cgd.parquet
 - multihiertt.parquet
 - nlvr2.parquet
 - ocrvqa.parquet
 - plotqa.parquet
 - raven.parquet
 - rendered_text.parquet
 - robut_sqa.parquet
 - robut_wikisql.parquet
 - robut_wtq.parquet
 - scienceqa.parquet
 - screen2words.parquet
 - semantic_evaluation.parquet
 - spot_the_diff.parquet
 - st_vqa.parquet
 - tabmwp.parquet
 - tallyqa.parquet
 - tat_qa.parquet
 - textcaps.parquet
 - textvqa.parquet
 - tqa.parquet
 - vistext.parquet
 - visual7w.parquet
 - visualmrc.parquet
 - vqarad.parquet
 - vqav2.parquet


In [ ]:


# ---- DB config ----
DB_USER = "vlmrouter"
DB_PASS = "vlmrouter"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "vlmrouter"
DB_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# ---- Build SQLAlchemy engine ----
engine = create_engine(
    DB_URL,
    echo=False,
)
inspector = inspect(engine)

# ---- Test connection ----
def test_connection():
    print("Connecting to PostgreSQL...")

    try:
        with engine.connect() as conn:
            result = conn.execute(text("SELECT NOW();"))
            row = result.fetchone()
            print("Connection successful!")
            print("Current time on DB:", row[0])
    except OperationalError as e:
        print("❌ Failed to connect to PostgreSQL")
        print(str(e))
test_connection()

Connecting to PostgreSQL...
Connection successful!
Current time on DB: 2025-12-05 21:19:33.869717+00:00


### Helper to derive table names from filenames

In [10]:
import re

def make_table_name_from_path(path):
    """
    E.g. 'router_train.parquet' -> 'cauldron_router_train'
    """
    stem = path.stem  # 'train', 'val', 'router_train', etc.
    safe = re.sub(r"[^0-9a-zA-Z_]", "_", stem).lower()
    table_name = f"cauldron_{safe}"
    return table_name


### Pre-Create Tables 

In [13]:
from tqdm import tqdm

for path in tqdm(PARQUET_FILES, desc="Pre-creating tables"):
    table_name = make_table_name_from_path(path)
    if inspector.has_table(table_name):
        print(f"Table '{table_name}' already exists, skipping create.")
        continue

    print(f"Creating table '{table_name}' from {path.name} schema...")
    df_head = pd.read_parquet(path, columns=None, engine="pyarrow").head(0)
    df_head.to_sql(
        table_name,
        engine,
        index=False,
        if_exists="fail",
    )

print("✅ Table creation step done.")


Pre-creating tables:   2%|▏         | 1/50 [00:00<00:06,  7.43it/s]

Creating table 'cauldron_ai2d' from ai2d.parquet schema...
Creating table 'cauldron_all_results' from all_results.parquet schema...


Pre-creating tables:  10%|█         | 5/50 [00:00<00:03, 14.28it/s]

Creating table 'cauldron_aokvqa' from aokvqa.parquet schema...
Creating table 'cauldron_chart2text' from chart2text.parquet schema...
Creating table 'cauldron_chartqa' from chartqa.parquet schema...
Creating table 'cauldron_clevr' from clevr.parquet schema...
Creating table 'cauldron_cocoqa' from cocoqa.parquet schema...
Creating table 'cauldron_datikz' from datikz.parquet schema...


Pre-creating tables:  26%|██▌       | 13/50 [00:00<00:01, 27.02it/s]

...

Creating table 'cauldron_scienceqa' from scienceqa.parquet schema...
Creating table 'cauldron_screen2words' from screen2words.parquet schema...
Creating table 'cauldron_semantic_evaluation' from semantic_evaluation.parquet schema...
Creating table 'cauldron_spot_the_diff' from spot_the_diff.parquet schema...
Creating table 'cauldron_st_vqa' from st_vqa.parquet schema...
Creating table 'cauldron_tabmwp' from tabmwp.parquet schema...
Creating table 'cauldron_tallyqa' from tallyqa.parquet schema...
Creating table 'cauldron_tat_qa' from tat_qa.parquet schema...


Pre-creating tables:  94%|█████████▍| 47/50 [00:01<00:00, 37.00it/s]

Creating table 'cauldron_textcaps' from textcaps.parquet schema...
Creating table 'cauldron_textvqa' from textvqa.parquet schema...
Creating table 'cauldron_tqa' from tqa.parquet schema...
Creating table 'cauldron_vistext' from vistext.parquet schema...
Creating table 'cauldron_visual7w' from visual7w.parquet schema...
Creating table 'cauldron_visualmrc' from visualmrc.parquet schema...
Creating table 'cauldron_vqarad' from vqarad.parquet schema...
Creating table 'cauldron_vqav2' from vqav2.parquet schema...


Pre-creating tables: 100%|██████████| 50/50 [00:01<00:00, 30.80it/s]

Creating table 'cauldron_vsr' from vsr.parquet schema...
Creating table 'cauldron_websight' from websight.parquet schema...
✅ Table creation step done.


#### Inspect the tables once: - 

In [17]:
import pandas as pd

def inspect_cauldron_tables(engine, prefix="cauldron_"):
    """
    List all tables whose name starts with `prefix` and print row counts.
    Returns a pandas DataFrame with (table_name, row_count).
    """
    insp = inspect(engine)
    table_names = [t for t in insp.get_table_names() if t.startswith(prefix)]

    if not table_names:
        print(f"No tables found with prefix '{prefix}'.")
        return pd.DataFrame(columns=["table_name", "row_count"])

    rows = []
    with engine.connect() as conn:
        for t in table_names:
            count = conn.execute(text(f"SELECT COUNT(*) FROM {t};")).scalar_one()
            print(f"{t}: {count} rows")
            rows.append({"table_name": t, "row_count": count})

    return pd.DataFrame(rows)


In [18]:
summary_df = inspect_cauldron_tables(engine, prefix="cauldron_")
summary_df

cauldron_ai2d: 0 rows
cauldron_all_results: 0 rows
cauldron_aokvqa: 0 rows
cauldron_chart2text: 0 rows
cauldron_chartqa: 0 rows
cauldron_clevr: 0 rows
cauldron_cocoqa: 0 rows
cauldron_datikz: 0 rows
cauldron_diagram_image_to_text: 0 rows
cauldron_docvqa: 0 rows
cauldron_dvqa: 0 rows
cauldron_figureqa: 0 rows
cauldron_finqa: 0 rows
cauldron_geomverse: 0 rows
cauldron_hateful_memes: 0 rows
cauldron_hitab: 0 rows
cauldron_iam: 0 rows
cauldron_iconqa: 0 rows
cauldron_infographic_vqa: 0 rows
cauldron_intergps: 0 rows
cauldron_localized_narratives: 0 rows
cauldron_mapqa: 0 rows
cauldron_mimic_cgd: 0 rows
cauldron_multihiertt: 0 rows
cauldron_nlvr2: 0 rows
cauldron_ocrvqa: 0 rows
cauldron_plotqa: 0 rows
cauldron_raven: 0 rows
cauldron_rendered_text: 0 rows
cauldron_robut_sqa: 0 rows
cauldron_robut_wikisql: 0 rows
cauldron_robut_wtq: 0 rows
cauldron_scienceqa: 0 rows
cauldron_screen2words: 0 rows
cauldron_semantic_evaluation: 0 rows
cauldron_spot_the_diff: 0 rows
cauldron_st_vqa: 0 rows
cauldr

,table_name,row_count
0,cauldron_ai2d,0
1,cauldron_all_results,0
2,cauldron_aokvqa,0
3,cauldron_chart2text,0
4,cauldron_chartqa,0
5,cauldron_clevr,0
6,cauldron_cocoqa,0
7,cauldron_datikz,0
8,cauldron_diagram_image_to_text,0
9,cauldron_docvqa,0


In [20]:
# def clear_cauldron_tables(engine, prefix="cauldron_"):
#     """
#     TRUNCATE all tables whose name starts with `prefix`.
#     Keeps table schemas, removes all rows.
#     """
#     insp = inspect(engine)
#     table_names = [t for t in insp.get_table_names() if t.startswith(prefix)]

#     if not table_names:
#         print(f"No tables found with prefix '{prefix}'. Nothing to clear.")
#         return

#     print("About to TRUNCATE these tables:")
#     for t in table_names:
#         print(" -", t)

#     # If you want an extra safety check, uncomment:
#     # confirm = input("Type 'yes' to confirm truncation: ")
#     # if confirm.lower() != "yes":
#     #     print("Aborted.")
#     #     return

#     with engine.begin() as conn:  # begin() handles commit/rollback
#         for t in table_names:
#             conn.execute(text(f"TRUNCATE TABLE {t};"))

#     print("✅ All matching tables truncated.")

# clear_cauldron_tables(engine, prefix="cauldron_")


### Ingest each split into its own table

In [23]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import json

MAX_WORKERS = 8  # you can tweak thi

In [24]:


def normalize_df_for_sql(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert problematic column types (numpy arrays, lists, dicts) into JSON strings
    so Postgres can store them as TEXT.
    Operates in-place and also returns df for convenience.
    """
    def convert_value(v):
        if isinstance(v, np.ndarray):
            return json.dumps(v.tolist())
        if isinstance(v, (list, tuple)):
            return json.dumps(list(v))
        if isinstance(v, dict):
            return json.dumps(v)
        return v

    for col in df.columns:
        # Only bother with object dtype columns
        if df[col].dtype == "object":
            sample = df[col].dropna().head(1)
            if sample.empty:
                continue
            v = sample.iloc[0]
            if isinstance(v, (np.ndarray, list, tuple, dict)):
                # Convert entire column
                df[col] = df[col].apply(
                    lambda x: convert_value(x) if x is not None else None
                )

    return df

def ingest_one_parquet_thread(path):
    """
    Thread worker: load one parquet file, normalize it, and append to its table.
    """
    path = Path(path)
    table_name = make_table_name_from_path(path)

    engine_local = create_engine(DB_URL, echo=False)

    df = pd.read_parquet(path)
    df = normalize_df_for_sql(df)  # <-- new line

    df.to_sql(
        table_name,
        engine_local,
        index=False,
        if_exists="append",
        chunksize=1000,
    )
    return path.name, len(df)



In [25]:
test_path = PARQUET_FILES[0]
df_test = pd.read_parquet(test_path)

for col in df_test.columns:
    if df_test[col].dtype == "object":
        s = df_test[col].dropna().head(1)
        if s.empty:
            continue
        v = s.iloc[0]
        if isinstance(v, (np.ndarray, list, tuple, dict)):
            print("Array-like column:", col, "example:", v)


Array-like column: mc_options example: ['(A) fewer benthic animals' '(B) fewer coyotes'
 '(C) fewer coyotes\nAnswer with the letter.']
Array-like column: glider_highlight example: ['C' 'The Sun' 'largest']


In [26]:

paths = [str(p) for p in PARQUET_FILES]

results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(ingest_one_parquet_thread, p): p
        for p in paths
    }

    for fut in tqdm(as_completed(futures), total=len(futures), desc="Ingesting parquet splits (threads)"):
        fname, nrows = fut.result()
        results.append((fname, nrows))

print("\n✅ Ingestion complete for all splits.")
for fname, nrows in results:
    print(f"{fname}: inserted {nrows} rows")

Ingesting parquet splits (threads): 100%|██████████| 50/50 [00:34<00:00,  1.46it/s]


✅ Ingestion complete for all splits.
chartqa.parquet: inserted 10000 rows
cocoqa.parquet: inserted 10000 rows
aokvqa.parquet: inserted 10000 rows
chart2text.parquet: inserted 10000 rows
clevr.parquet: inserted 10000 rows
datikz.parquet: inserted 10000 rows
ai2d.parquet: inserted 10000 rows
diagram_image_to_text.parquet: inserted 1500 rows
dvqa.parquet: inserted 10000 rows
figureqa.parquet: inserted 10000 rows
docvqa.parquet: inserted 10000 rows
finqa.parquet: inserted 10000 rows
hateful_memes.parquet: inserted 10000 rows
geomverse.parquet: inserted 10000 rows
hitab.parquet: inserted 10000 rows
iam.parquet: inserted 10000 rows
intergps.parquet: inserted 6400 rows
iconqa.parquet: inserted 10000 rows
infographic_vqa.parquet: inserted 10000 rows
localized_narratives.parquet: inserted 9990 rows
mapqa.parquet: inserted 10000 rows
mimic_cgd.parquet: inserted 10000 rows
nlvr2.parquet: inserted 10000 rows
multihiertt.parquet: inserted 10000 rows
ocrvqa.parquet: inserted 10000 rows
plotqa.parqu

In [27]:
with engine.connect() as conn:
    for path in PARQUET_FILES:
        table_name = make_table_name_from_path(path)
        result = conn.execute(text(f"SELECT COUNT(*) FROM {table_name};"))
        count = result.scalar_one()
        print(f"Table {table_name}: {count} rows")


Table cauldron_ai2d: 10000 rows
Table cauldron_all_results: 77450 rows
Table cauldron_aokvqa: 10000 rows
Table cauldron_chart2text: 10000 rows
Table cauldron_chartqa: 10000 rows
Table cauldron_clevr: 10000 rows
Table cauldron_cocoqa: 10000 rows
Table cauldron_datikz: 10000 rows
Table cauldron_diagram_image_to_text: 1500 rows
Table cauldron_docvqa: 10000 rows
Table cauldron_dvqa: 10000 rows
Table cauldron_figureqa: 10000 rows
Table cauldron_finqa: 10000 rows
Table cauldron_geomverse: 10000 rows
Table cauldron_hateful_memes: 10000 rows
Table cauldron_hitab: 10000 rows
Table cauldron_iam: 10000 rows
Table cauldron_iconqa: 10000 rows
Table cauldron_infographic_vqa: 10000 rows
Table cauldron_intergps: 6400 rows
Table cauldron_localized_narratives: 9990 rows
Table cauldron_mapqa: 10000 rows
Table cauldron_mimic_cgd: 10000 rows
Table cauldron_multihiertt: 10000 rows
Table cauldron_nlvr2: 10000 rows
Table cauldron_ocrvqa: 10000 rows
Table cauldron_plotqa: 10000 rows
Table cauldron_raven: 10000

In [28]:
split_path = PARQUET_FILES[0]  # or pick by index/name
table_name = make_table_name_from_path(split_path)

df_preview = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 5;", engine)
df_preview


,sample_id,run_id,timestamp_utc,image_path,image_bytes_hash,prompt_raw,prompt_formatted,system_prompt,source_dataset,source_config,...,semantic_f1_recall,semantic_f1_f1,semantic_f1_gen_statements,semantic_f1_gt_statements,semantic_f1_matches,semantic_f1_labels,glider_score,glider_reasoning,glider_highlight,glider_raw_output
0,ai2d_00024_ab0c600e6d10613f,exp_20251127_132944,2025-11-27T18:30:57.471038,None,ab0c600e6d10613f,Question: Which of the above things are the la...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,None,5,"- The model's output ""C"" is the correct answer...","[""C"", ""The Sun"", ""largest""]","<reasoning>\n- The model's output ""C"" is the ..."
1,ai2d_00025_825ad24fd4599ce7,exp_20251127_132944,2025-11-27T18:31:03.026694,None,825ad24fd4599ce7,Question: What is the layer above the upper ma...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,None,0,- The model's output does not provide any answ...,"[""does not provide"", ""expected answer"", ""corre...",<reasoning>\n- The model's output does not pr...
2,ai2d_00026_baeba96290d54334,exp_20251127_132944,2025-11-27T18:31:08.863450,None,baeba96290d54334,Question: What phase is shown above\nChoices:\...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,None,0,"- The model's output ""D"" is incorrect as it co...","[""D"", ""photosynthesis"", ""A""]","<reasoning>\n- The model's output ""D"" is inco..."
3,ai2d_00027_658d245983f8af4c,exp_20251127_132944,2025-11-27T18:31:16.092776,None,658d245983f8af4c,Question: What comes after Pupa?\nChoices:\nA....,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,None,0,"- The model's output ""C. Eggs"" is incorrect as...","[""C. Eggs"", ""Answer: B"", ""Pupa"", ""Adult""]","<reasoning>\n- The model's output ""C. Eggs"" i..."
4,ai2d_00028_e1bae1fe1bb31d0d,exp_20251127_132944,2025-11-27T18:31:22.193315,None,e1bae1fe1bb31d0d,Question: How many different types of leaves w...,None,None,cauldron_ai2d,ai2d,...,None,None,None,None,None,None,0,"- The model's output ""D"" is incorrect as it co...","[""D"", ""7"", ""8""]","<reasoning>\n- The model's output ""D"" is inco..."
